## Cluste 1fps extractions

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

import numpy as np
import fiftyone as fo
import fiftyone.zoo as foz
import fiftyone.brain as fob


In [ ]:

PROJECT_NAME = "arsenal_mancity_frames_1fps"
DATASET_DIR = Path("../data/arsenal_mancity_frames_1fps")


In [ ]:

if False and fo.dataset_exists(PROJECT_NAME):
    fo.delete_dataset(PROJECT_NAME)

if fo.dataset_exists(PROJECT_NAME):
    dataset = fo.load_dataset(PROJECT_NAME)
else:
    dataset = fo.Dataset.from_images_dir(DATASET_DIR, name= PROJECT_NAME,overwrite=False, persistent=True, progress=True)


In [ ]:
session = fo.launch_app(dataset, port=5151)

In [ ]:
# Compute and attach embeddings to each sample
from pathlib import Path
import numpy as np

from footy_track import frame_embedding, object_detections

# extractor = frame_embedding.ResNet50FeatureExtractor()
embedder = frame_embedding.ConvNeXtBaseFeatureExtractor()
detector = object_detections.UltralyticsObjectDetector()

embedding_field_name = "embedding"
detection_field_name = "detections"

if False: 
    for sample in dataset.iter_samples(progress=True):

        vec = embedder.extract_from_path(Path(sample.filepath))
        sample[embedding_field_name] = np.asarray(vec, dtype=np.float32)

        
        detection_result = detector.predict_from_path(Path(sample.filepath))
        fo_detections = object_detections.ultralytics_detection_to_fiftyone_detection(detection_result, classes=list(detector.classes.values()))
        sample[detection_field_name] = fo.Detections(detections=fo_detections)

        sample.save()

    print(f"Attached embeddings to {len(dataset)} samples")

In [ ]:
sample = dataset.first()
detection_result = detector.predict_from_path(Path(sample.filepath))


## Visualize embeddings
Compute a 2D projection (UMAP with fallback to PCA) and visualize the frames.

- Uses FiftyOne Brain for interactive visualization in the App (Spaces tab)
- Also stores 2D coords as `viz_x`, `viz_y` on each sample for ad-hoc plotting

In [ ]:
# Compute 2D projection and visualize in FiftyOne App
import numpy as np
import fiftyone.brain as fob

# Ensure we have embeddings on all samples
assert dataset.exists("embedding"), "Embeddings missing; run the previous cell first."


results = fob.compute_visualization(
    dataset,
    embeddings=embedding_field_name,
    method="umap",
    brain_key="embeddings",
    seed=42,
)

## Add a classification label field: `correct_view`
Create a new Classification field on each sample to mark whether the frame is the correct camera/view. You can edit this field directly in the FiftyOne App (Samples panel > Fields sidebar) by setting `correct_view.label` to values like `correct` or `incorrect`.

In [ ]:


# # Basic KMeans clustering on frame embeddings
# import numpy as np
# from collections import Counter

# try:
#     from sklearn.cluster import KMeans, HDBSCAN
# except Exception as e:
#     raise RuntimeError(
#         "scikit-learn is required for clustering. Install it via `pip install scikit-learn` and re-run this cell."
#     ) from e

# # Ensure embeddings are present
# assert dataset.exists(embedding_field_name), f"Embeddings field '{embedding_field_name}' not found"

# # Collect embeddings and corresponding sample ids
# sample_ids = []
# embeddings = []
# for sample in dataset.iter_samples(progress=True):
#     vec = sample[embedding_field_name]
#     if vec is not None:
#         sample_ids.append(sample.id)
#         embeddings.append(np.asarray(vec, dtype=np.float32))

# if not embeddings:
#     raise ValueError("No embeddings found to cluster")

# X = np.stack(embeddings)

# # Choose number of clusters (simple heuristic with bounds)
# n_clusters = min(12, max(2, int(np.sqrt(len(X) / 2))))
# n_clusters = 4

# # clusterer = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
# clusterer = HDBSCAN(min_cluster_size=20)
# labels = clusterer.fit_predict(X)

# # Add integer field for cluster index
# cluster_field = "cluster_id"
# if not dataset.has_sample_field(cluster_field):
#     dataset.add_sample_field(cluster_field, fo.IntField, description="KMeans cluster index")

# labels_by_id = dict(zip(sample_ids, labels.tolist()))
# for sample in dataset.iter_samples(progress=True):
#     lbl = labels_by_id.get(sample.id)
#     if lbl is not None:
#         sample[cluster_field] = int(lbl)
#         sample.save()

# # Print summary
# counts = Counter(labels.tolist())
# print(f"Clustered {len(X)} samples into {n_clusters} clusters")
# for cid in sorted(counts):
#     print(f"Cluster {cid}: {counts[cid]} samples")

# # Update App view to sort by cluster (optional)
# try:
#     session.view = dataset.sort_by(cluster_field)
# except Exception:
#     pass

## Export sample tags to JSON
This exports a JSON file listing each sample's file path and any of the allowed tags: `broadcast`, `non-broadcast`, or `overhead`. Run the next cell after you've applied tags in the FiftyOne App.

In [ ]:
dataset.export(
    export_dir="data/export",
    dataset_type=fo.types.COCODetectionDataset,
)

In [ ]:
# Export sample tags to JSON (path -> tags)
import json
from pathlib import Path

# Allowed/expected tag categories
ALLOWED_TAGS = {"broadcast", "non-broadcast", "overhead"}

# Output path (relative to notebook folder)
OUTPUT_JSON = Path("../data/arsenal_mancity_frames_1fps_tags.json").resolve()

export = {}
for sample in dataset.iter_samples(progress=True):
    tags = list(set(sample.tags or []))
    export[sample.filepath] = {
        "tags": tags,
        "detections": [det for det in sample.detections.detections] if sample.detections else []
    }

with OUTPUT_JSON.open("w") as f:
    json.dump(export, f, indent=4)

In [ ]:
OUTPUT_JSON = Path("../data/arsenal_mancity_frames_1fps_tags.json").resolve()

with OUTPUT_JSON.open("w") as f:
    loaded_data = json.load(f)

new_export = {}
for sample in loaded_data:
    tags = list(set(sample.get("tags", [])))
    new_export[sample.get("filepath")] = {
        "tags": tags,
        "detections": sample.get("detections", []),
        "filepath": sample.get("filepath")
    }



In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")  # load an official model

# Visualise eaxmple model

In [ ]:
from IPython.display import display
from PIL import Image

first_sample = dataset.first()

middle_index = len(dataset) // 2
view = dataset.skip(middle_index).take(1)
sample = view.first()
results = model(sample.filepath)
im_bgr = results[0].plot()  # BGR-order numpy array
im_rgb = Image.fromarray(im_bgr[..., ::-1])  # RGB-order PIL image
display(im_rgb)

In [ ]:
samples = [sample for sample in dataset.iter_samples(progress=True)]

filepaths = [s.filepath for s in samples]
batch_size = 128

for i in range(0, len(filepaths), batch_size):
    batch_paths = filepaths[i : i + batch_size]
    batch_results = model(batch_paths, device="mps")
    for j, sample in enumerate(samples[i : i + batch_size]):
        results = batch_results[j]
        pass
    

In [ ]:

# with open(OUTPUT_JSON, "w") as f:
#     json.dump(export, f, indent=2)

# print(f"Wrote {len(export)} records to {OUTPUT_JSON}")
